# LPatchTST — Kaggle 2x T4 GPU Notebook

This notebook handles repository setup, data preparation, multi-GPU DDP training, and downstream evaluation on Kaggle.

### Pre-requisites:
1. **Internet** must be turned **ON** in the right-hand settings panel.
2. **Accelerator** must be set to **GPU T4 x2**.

In [28]:
%%bash
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone Repo & Set Up CSV Dataset                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
REPO_DIR="/kaggle/working/Lpatchtst"

if [ -d "$REPO_DIR" ]; then
    echo "Repository already exists. Updating to latest main..."
    cd "$REPO_DIR"
    git fetch origin
    git reset --hard origin/main
    git submodule update --init --recursive
else
    echo "Cloning repository (including submodules)..."
    git clone --recurse-submodules https://github.com/ayan1-git/Lpatchtst "$REPO_DIR"
fi

KRONOS_DIR="/kaggle/working/Kronos_finetune"
if [ -d "$KRONOS_DIR" ]; then
    echo "Kronos_finetune repository already exists. Updating..."
    cd "$KRONOS_DIR"
    git fetch origin
    git reset --hard origin/main
else
    echo "Cloning Kronos_finetune repository..."
    git clone https://github.com/ayan1-git/Kronos_finetune "$KRONOS_DIR"
fi

# Install lightweight required libraries (torch/numpy/pandas are pre-installed)
pip install -q einops safetensors scikit-learn

cd "$REPO_DIR"
mkdir -p Data

# Check if the cloned repository already contains CSV files in Data/
csv_count=$(ls Data/*.csv 2>/dev/null | wc -l)

if [ "$csv_count" -gt 0 ]; then
    echo "Found $csv_count pre-packaged CSV files in Data/ directory (cloned from Git)."
    echo "Using pre-packaged dataset for training."
    ls -lh Data/
else
    # Auto-discover any Kaggle input folder containing CSV files and link them
    echo "No CSV files found in cloned Data/ folder. Searching under /kaggle/input/..."
    FOUND=0
    for d in /kaggle/input/*; do
        if [ -d "$d" ]; then
            if ls "$d"/*.csv >/dev/null 2>&1; then
                echo "-> Found CSV files in $d. Symlinking to Data/..."
                ln -sf "$d"/*.csv Data/
                FOUND=1
            fi
        fi
    done
    if [ $FOUND -eq 0 ]; then
        echo "⚠️ Warning: No CSV files found. Please copy your CSVs to $REPO_DIR/Data/."
    else
        echo "Data folder contents successfully mapped from Kaggle inputs:"
        ls -lh Data/
    fi
fi
echo "✅ Setup complete!"

Repository already exists. Updating to latest main...
HEAD is now at 64b4fca ch
Kronos_finetune repository already exists. Updating...
HEAD is now at 7108450 ch
Found 19 pre-packaged CSV files in Data/ directory (cloned from Git).
Using pre-packaged dataset for training.
total 28M
-rw-r--r-- 1 root root 1.9M May 31 04:21 NIFTY 100_30minute.csv
-rw-r--r-- 1 root root 1.9M May 31 04:21 NIFTY 200_30minute.csv
-rw-r--r-- 1 root root 1.2M May 31 04:21 NIFTY 500_30minute.csv
-rw-r--r-- 1 root root 1.9M May 31 04:21 NIFTY 50_30minute.csv
-rw-r--r-- 1 root root 1.2M May 31 04:21 NIFTY ALPHA 50_30minute.csv
-rw-r--r-- 1 root root 1.9M May 31 04:21 NIFTY AUTO_30minute.csv
-rw-r--r-- 1 root root 2.0M May 31 04:21 NIFTY BANK_30minute (1).csv
-rw-r--r-- 1 root root 1.7M May 31 04:21 NIFTY COMMODITIES_30minute.csv
-rw-r--r-- 1 root root 633K May 31 04:21 NIFTY CONSR DURBL_30minute.csv
-rw-r--r-- 1 root root 1.4M May 31 04:21 NIFTY CONSUMPTION_30minute.csv
-rw-r--r-- 1 root root 1.1M May 31 04:21 NIF

From https://github.com/ayan1-git/Lpatchtst
   94d0f39..64b4fca  main       -> origin/main


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Run DDP Training on 2x T4 GPUs via torchrun               ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys, torch

REPO_DIR = "/kaggle/working/Lpatchtst"
os.chdir(REPO_DIR)

# 1. Force unbuffered output and resolve NCCL binding hangs in subprocesses
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["NCCL_SOCKET_IFNAME"] = "lo"

num_gpus = torch.cuda.device_count()
print(f"Detecting GPUs: {num_gpus} active")

cmd = f"torchrun --standalone --nnodes=1 --nproc_per_node={num_gpus} --master_port=29500 train.py"
print(f"Running command: {cmd}\n")
print("-" * 70)

# 2. Launch process and stream stdout+stderr line-by-line in real-time
process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=REPO_DIR
)

try:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Terminating training process…")
    process.terminate()
    process.wait()

rc = process.poll()
if rc != 0:
    raise RuntimeError(f"Training failed with exit code {rc}")
print("\n✅ Training complete successfully!")

Detecting GPUs: 2 active
Running command: torchrun --standalone --nnodes=1 --nproc_per_node=2 --master_port=29500 finetune_tokenizer.py

----------------------------------------------------------------------
W0531 08:49:20.316000 1310 torch/distributed/run.py:852] 
W0531 08:49:20.316000 1310 torch/distributed/run.py:852] *****************************************
W0531 08:49:20.316000 1310 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0531 08:49:20.316000 1310 torch/distributed/run.py:852] *****************************************
[W531 08:49:20.162486263 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W531 08:49:23.683715902 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W531 08:49:23.691583343 socket.cpp:207] [c10d] T

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Downstream Evaluation & Diagnostics                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys

REPO_DIR = "/kaggle/working/Lpatchtst"
os.chdir(REPO_DIR)

cmd = "python3 -u evaluate.py"
print(f"Running command: {cmd}\n")
print("-" * 70)

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR
)

try:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Terminating evaluation process…")
    process.terminate()
    process.wait()

rc = process.poll()
if rc != 0:
    raise RuntimeError(f"Evaluation failed with exit code {rc}")
print("\n✅ Evaluation complete successfully!")

Running command: python3 -u prepare_niklib.py

----------------------------------------------------------------------
train: 19 symbols saved
val: 19 symbols saved
test: 19 symbols saved

✅ Evaluation complete successfully!
